## Models Comparision
---
This notebook will show case different models performance for recommendation system using machine learning algorithms. To compare the model's performance the metrics we are using are accruracy_score, f1_score, classification_report and confusion_matrix with precision and recall. 

In [50]:
# Necessary imports

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [65]:
# one hot encoding function
import sqlite3
import pandas as pd
import numpy as np
import ast

def ohe_column_encoding(hot: list, df: pd.DataFrame):
    
    if hot is not None:
        ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False).set_output(transform="pandas")

        for col in hot:
            temp = ohe.fit_transform(df[[col]])
            df = pd.concat([df, temp], axis=1).drop(columns=[col])
    
        return df
    
    return df


def load_training_dataset():

    conn=sqlite3.connect(
        "../experiment.db"
    )

    df=pd.read_sql(
        "SELECT * FROM dataset where label=0 or label=2",
        conn
    )

    conn.close()

    json_cols=[

    "subjects",
    "interests",
    "preferred_skills",

    "skills_learned",

    "career_outcomes",

    "required_subjects"

    ]

    for col in json_cols:

        df[col] = df[col].apply(lambda x: [] if pd.isna(x) else ast.literal_eval(x))




    bool_cols=[

    "is_reserved",

    "domain_match",

    "subject_required",

    "marks_required"

    ]

    for col in bool_cols:

        df[col]=df[col].astype(bool)


    numeric_cols=[

    "percentage",

    "similarity_score",

    "subject_overlap",

    "marks_margin",

    "skill_relevance_percentage",

    "career_align_percentage",

    "label"

    ]

    for col in numeric_cols:

        df[col]=pd.to_numeric(
            df[col],
            errors="coerce"
        )
    df['min_marks_general'] = (
        df["min_marks_general"].fillna(-1)
    )
    df['min_marks_reserved'] = (
        df["min_marks_reserved"].fillna(-1)
    )
    df['marks_margin'] = np.where(
        df['marks_required'] == 0,
        -1,
        df['marks_margin']
    )
    df['duration'] = (
        df['duration'].fillna("N/A")
    )
    return df


# Function to evaluate the results
def evaluate_results(predictions, y_test):
    print("\nAccuracy: \n")
    print(accuracy_score(y_test, predictions))

    print("\nF1 scores:\n")
    print(f1_score(y_test, predictions, average="macro"))

    print("\nClassification Report: \n")
    print(classification_report(y_test, predictions))

    # Confustion Matrix
    cm = confusion_matrix(y_test, predictions)
    print(cm)

In [66]:
# Loading and preparing dataset
dataset = load_training_dataset()

features = [

    "similarity_score",
    "domain_match",
    "subject_required",
    "subject_overlap",
    "marks_required",
    "marks_margin",
    "skill_relevance_percentage",
    "career_align_percentage"
]

y = dataset.pop("label")
X = dataset[features].copy()


In [74]:
y = y.apply(lambda x: 1 if x == 2 else 0)
y.value_counts()

label
0    289
1    190
Name: count, dtype: int64

In [78]:
# Initialising the models to compare
logistic_model_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=500))
])

random_forest_model = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)

xgboost_model = XGBClassifier(
    n_estimator=200,
    max_depth=6,
    subsample=0.8,
    learning_rate = 0.05,
    colsample_bytree=0.8,
    objective="binary:logistic",
    num_classes=2,
    random_state=42
)

lgbm_model = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    random_state=42
)


In [79]:
# Performing the train and test split on dataset
 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=42, stratify=y)

In [80]:
# Initailising models

logistic_model_pipeline.fit(X_train, y_train)
random_forest_model.fit(X_train, y_train)
xgboost_model.fit(X_train, y_train)
lgbm_model.fit(X_train, y_train)

[15:57:50] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "n_estimator", "num_classes" } are not used.



[LightGBM] [Info] Number of positive: 152, number of negative: 231
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000420 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 400
[LightGBM] [Info] Number of data points in the train set: 383, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.396867 -> initscore=-0.418537
[LightGBM] [Info] Start training from score -0.418537
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,6
,learning_rate,0.05
,n_estimators,200
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [81]:
log_pred = logistic_model_pipeline.predict(X_test)
rf_pred = random_forest_model.predict(X_test)
xg_pred = xgboost_model.predict(X_test)
lgb_pred = lgbm_model.predict(X_test)

In [82]:
print(f"=====================   Logistic Regression  =====================")
evaluate_results(log_pred, y_test)
print("\n\n")
print(f"=====================   Random Forest Classifier  =====================")
evaluate_results(rf_pred, y_test)
print("\n\n")
print(f"=====================   XGBoost Classifier  =====================")
evaluate_results(xg_pred, y_test)
print("\n\n")
print(f"=====================   LightGBM Classifier  =====================")
evaluate_results(lgb_pred, y_test)

=====================   Logistic Regression  =====================

Accuracy: 

0.8645833333333334

F1 scores:

0.8547316959608893

Classification Report: 

              precision    recall  f1-score   support

           0       0.86      0.93      0.89        58
           1       0.88      0.76      0.82        38

    accuracy                           0.86        96
   macro avg       0.87      0.85      0.85        96
weighted avg       0.87      0.86      0.86        96

[[54  4]
 [ 9 29]]



=====================   Random Forest Classifier  =====================

Accuracy: 

0.8645833333333334

F1 scores:

0.8590626764539808

Classification Report: 

              precision    recall  f1-score   support

           0       0.89      0.88      0.89        58
           1       0.82      0.84      0.83        38

    accuracy                           0.86        96
   macro avg       0.86      0.86      0.86        96
weighted avg       0.87      0.86      0.86        96

[[51 

These are the results of the models results including the general accuracy, f1_score, classification report and confusion_matrix.

---

```text
=====================   Logistic Regression  =====================

Accuracy: 

0.5416666666666666

F1 scores:

0.5363728283849619

Classification Report: 

              precision    recall  f1-score   support

           0       0.55      0.62      0.58        26
           1       0.53      0.35      0.42        26
           2       0.54      0.70      0.61        20

    accuracy                           0.54        72
   macro avg       0.54      0.55      0.54        72
weighted avg       0.54      0.54      0.53        72

[[16  5  5]
 [10  9  7]
 [ 3  3 14]]



=====================   Random Forest Classifier  =====================

Accuracy: 

0.625

F1 scores:

0.6282568122433744

Classification Report: 

              precision    recall  f1-score   support

           0       0.58      0.69      0.63        26
           1       0.62      0.50      0.55        26
           2       0.70      0.70      0.70        20

    accuracy                           0.62        72
   macro avg       0.63      0.63      0.63        72
weighted avg       0.63      0.62      0.62        72

[[18  5  3]
 [10 13  3]
 [ 3  3 14]]



=====================   XGBoost Classifier  =====================

Accuracy: 

0.6527777777777778

F1 scores:

0.6544516371141759

Classification Report: 

              precision    recall  f1-score   support

           0       0.66      0.73      0.69        26
           1       0.60      0.58      0.59        26
           2       0.72      0.65      0.68        20

    accuracy                           0.65        72
   macro avg       0.66      0.65      0.65        72
weighted avg       0.65      0.65      0.65        72

[[19  5  2]
 [ 8 15  3]
 [ 2  5 13]]



=====================   LightGBM Classifier  =====================

Accuracy: 

0.6111111111111112

F1 scores:

0.6131276467029644

Classification Report: 

              precision    recall  f1-score   support

           0       0.59      0.73      0.66        26
           1       0.55      0.46      0.50        26
           2       0.72      0.65      0.68        20

    accuracy                           0.61        72
   macro avg       0.62      0.61      0.61        72
weighted avg       0.61      0.61      0.61        72

[[19  5  2]
 [11 12  3]
 [ 2  5 13]]
```

In the next cell the datset inludes encoded categorical features to see any potential improvements in the accuracy. Starting off with domain and preferred_domain.

In [58]:
# Reloading the dataset
dataset = load_training_dataset()
features.append('domain')
features.append('preferred_domain')
y = dataset.pop("label")
X = dataset[features].copy()

# Performing One hot encoding on domain and preferred_domain
domains = ['Science', 'Engineering', 'Commerce', 'Management', 'Humanities', 'Law', 'Arts', 'Environment']
encoder = OneHotEncoder(categories=[domains], sparse_output=False).set_output(transform='pandas')
enc_1 = encoder.fit_transform(dataset[['domain']])
enc_2 = encoder.fit_transform(dataset[['preferred_domain']])
X = pd.concat([X, enc_1, enc_2], axis=1).drop(columns=['domain', 'preferred_domain'])


In [59]:
# Doing the train test split again on new features included
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=42, stratify=y)

In [60]:
# Retraining the models
logistic_model_pipeline.fit(X=X_train, y=y_train)
random_forest_model.fit(X=X_train, y=y_train)
xgboost_model.fit(X=X_train, y=y_train)
lgbm_model.fit(X=X_train, y=y_train)

[00:10:21] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "n_estimator", "num_classes" } are not used.



[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000255 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 654
[LightGBM] [Info] Number of data points in the train set: 564, number of used features: 21
[LightGBM] [Info] Start training from score -0.892637
[LightGBM] [Info] Start training from score -1.136557
[LightGBM] [Info] Start training from score -1.311174
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,6
,learning_rate,0.05
,n_estimators,200
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [61]:
log_pred = logistic_model_pipeline.predict(X=X_test)
rf_pred = random_forest_model.predict(X=X_test)
xg_pred = xgboost_model.predict(X=X_test)
lgb_pred = lgbm_model.predict(X=X_test)

In [62]:
# Evaluating results with new features added
print(f"=====================   Logistic Regression  =====================")
evaluate_results(log_pred, y_test)
print("\n\n")
print(f"=====================   Random Forest Classifier  =====================")
evaluate_results(rf_pred, y_test)
print("\n\n")
print(f"=====================   XGBoost Classifier  =====================")
evaluate_results(xg_pred, y_test)
print("\n\n")
print(f"=====================   LightGBM Classifier  =====================")
evaluate_results(lgb_pred, y_test)

=====================   Logistic Regression  =====================

Accuracy: 

0.5957446808510638

F1 scores:

0.5796803528146812

Classification Report: 

              precision    recall  f1-score   support

           0       0.59      0.78      0.67        58
           1       0.48      0.36      0.41        45
           2       0.72      0.61      0.66        38

    accuracy                           0.60       141
   macro avg       0.60      0.58      0.58       141
weighted avg       0.59      0.60      0.58       141

[[45 11  2]
 [22 16  7]
 [ 9  6 23]]



=====================   Random Forest Classifier  =====================

Accuracy: 

0.5886524822695035

F1 scores:

0.5708995715654771

Classification Report: 

              precision    recall  f1-score   support

           0       0.63      0.76      0.69        58
           1       0.44      0.36      0.40        45
           2       0.66      0.61      0.63        38

    accuracy                           0.5

In [83]:
# Training and evaluation using cross eval score instead of train test split

models = [
    ("Logistic Regression", logistic_model_pipeline),
    ("Random Forest", random_forest_model),
    ("XGBoost", xgboost_model),
    ("LightGBM", lgbm_model)
]

for name, model in models:
    scores = cross_val_score(
        model,
        X=X,
        y=y,
        cv=5,
        scoring='f1_macro'
    )
    print(f"\n{name}")
    print(f"Mean: {scores.mean():.4f}")
    print(f"Std:  {scores.std():.4f}")


Logistic Regression
Mean: 0.7797
Std:  0.0501

Random Forest
Mean: 0.7879
Std:  0.0550


[17:08:54] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "n_estimator", "num_classes" } are not used.

[17:08:54] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "n_estimator", "num_classes" } are not used.

[17:08:54] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "n_estimator", "num_classes" } are not used.

[17:08:54] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "n_estimator", "num_classes" } are not used.

[17:08:54] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "n_estimator", "num_classes" } are not used.




XGBoost
Mean: 0.7951
Std:  0.0459
[LightGBM] [Info] Number of positive: 152, number of negative: 231
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000301 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 397
[LightGBM] [Info] Number of data points in the train set: 383, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.396867 -> initscore=-0.418537
[LightGBM] [Info] Start training from score -0.418537
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits w

In [85]:
import joblib

joblib.dump(logistic_model_pipeline, "../models/lr_model.pkl")
joblib.dump(random_forest_model, "../models/rf_model.pkl")
joblib.dump(xgboost_model, "../models/xg_model.pkl")
joblib.dump(lgbm_model, "../models/lgbm_model.pkl")

print("Models trained and pickled successfully...")

Models trained and pickled successfully...
